# FoodTrip — Train đọc hiểu, FAQ và sinh câu trả lời

Chọn T4 GPU. Kết nối runtime, mở thanh **Files → Upload to session storage**, tải cả `foodtrip-assistant-colab.zip` và `foodtrip-generator-colab.zip`. Sau đó Run all. Không cần khóa API thương mại.

Đây là đợt huấn luyện ban đầu: kết quả loss không chứng minh chất lượng hội thoại. Cần đọc báo cáo và kiểm tra câu trả lời trước khi đưa lên website.

## 1. Nạp hai bộ train đã tải qua thanh Files
Tệp không chứa .env hoặc dữ liệu khách hàng. Chỉ có FAQ công khai và dữ liệu mẫu.

In [ ]:
from google.colab import files
from pathlib import Path
import json, zipfile
WORK = Path('/content/foodtrip-assistant')
WORK.mkdir(exist_ok=True)
if not Path('/content/foodtrip-generator-colab.zip').exists():
    raise RuntimeError('Tải cả hai tệp ZIP qua Files trước khi Run all')
with zipfile.ZipFile('/content/foodtrip-assistant-colab.zip') as archive:
    for item in archive.infolist():
        if not (WORK / item.filename).resolve().is_relative_to(WORK.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(WORK)
print('TRAIN_DATA_READY')

## 2. Môi trường GPU riêng
Cài Python 3.12 và thư viện trong venv riêng (giống notebook NER). Chờ **GPU_SETUP_OK** rồi mới chạy tiếp.

In [ ]:
"""Paste this entire file into Colab's GPU environment cell (not a local run)."""
import os
import subprocess
import sys
from pathlib import Path


def run(command, *, cwd=None, env=None):
    """Stream both output channels so installation failures are visible in Colab."""
    process_env = dict(os.environ if env is None else env)
    process_env['PYTHONUNBUFFERED'] = '1'
    with subprocess.Popen(command, cwd=cwd, env=process_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)


WORK = Path('/content/foodtrip-assistant')
if not (WORK / 'requirements-colab.txt').is_file():
    raise RuntimeError('Chạy cell upload foodtrip-assistant-colab.zip trước')
run(['nvidia-smi'])
TOOLS = Path('/content/foodtrip-uv-tools')
run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
     '--target', str(TOOLS), '--upgrade', 'uv==0.9.5'])
uv_env = dict(os.environ)
uv_env['PYTHONPATH'] = str(TOOLS)
UV = [sys.executable, '-m', 'uv']
run(UV + ['python', 'install', '3.12'], env=uv_env)
# A different path preserves the incomplete environment from the failed cell.
ENV = Path('/content/foodtrip-assistant-py312')
PY = str(ENV / 'bin/python')
if not Path(PY).exists():
    run(UV + ['venv', '--python', '3.12', '--seed', str(ENV)], env=uv_env)
run([PY, '-c', 'import sys; print(sys.version); assert sys.version_info[:2] == (3, 12)'])
run([PY, '-m', 'pip', 'install', 'pip==25.0.1', 'setuptools==75.8.0', 'wheel', 'numpy==2.2.6'])
run([PY, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run([PY, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(WORK / 'requirements-colab.txt')])
run([PY, '-m', 'pip', 'check'])
run([PY, '-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Chưa có GPU'; print(torch.cuda.get_device_name(0))"])
print('GPU_SETUP_OK — tiếp tục mục 3. Kiểm tra và smoke test', flush=True)

## 3. Kiểm tra và smoke test
Kiểm tra mã hóa/nhãn và dữ liệu, rồi chạy thử 3 bước train — **chưa phải train thật**. Có lỗi thì dừng.

In [ ]:
run([PY, str(WORK / "test_assistant.py"), "data"], cwd=WORK)
run([PY, str(WORK / "train_assistant.py"), "--data", "data", "--output", "runs/smoke", "--smoke"], cwd=WORK)
run([PY, str(WORK / "train_retriever.py"), "--data", "data", "--output", "runs/retriever-smoke", "--smoke"], cwd=WORK)

## 4. Train
Tối đa 10 epoch, dừng sớm theo validation (trung bình độ đúng ý định và F1 thông tin). Trên T4 thường khoảng 10–20 phút.

In [ ]:
if not (WORK / "runs/smoke/smoke-passed.json").exists():
    raise RuntimeError("Smoke test chưa qua")
run([PY, str(WORK / "train_assistant.py"), "--data", "data", "--output", "runs/assistant", "--export-onnx"], cwd=WORK)

## 5. Kết quả
So với bộ luật trên tập test (cách nói chưa gặp khi train) và 100 câu viết tay. Không chỉnh mô hình theo tập test.

In [ ]:
RESULT = WORK / "runs/assistant"
print((RESULT / "REPORT.md").read_text())
errors = json.loads((RESULT / "errors.json").read_text())
print(len(errors), "câu sai (ý định hoặc thông tin). 15 câu đầu:")
for e in errors[:15]:
    print("-", e["text"], "| đúng:", e["intent"], "| đoán:", e["predicted"])

## 5b. Train bộ tìm câu trả lời
Tinh chỉnh bộ mã hóa câu tiếng Việt để mỗi câu hỏi tìm đúng mục hỏi–đáp của FoodTrip. Vài phút trên T4. So với cách so khớp từ ngữ đang chạy và với bộ mã hóa chưa tinh chỉnh.

In [ ]:
if not (WORK / "runs/retriever-smoke/smoke-passed.json").exists():
    raise RuntimeError("Smoke test của bộ tìm câu trả lời chưa qua")
run([PY, str(WORK / "train_retriever.py"), "--data", "data", "--output", "runs/retriever", "--export-onnx"], cwd=WORK)
print((WORK / "runs/retriever/RETRIEVER.md").read_text())

## 6. Thử một câu

In [ ]:
run([PY, str(WORK / "assistant_infer.py"), str(RESULT / "onnx"), "cuối tuần này đi Đà Lạt 2 người, 3 triệu, thích cà phê"], cwd=WORK)

## 7. Tải mô hình về
Gói gồm hai mô hình ONNX (đọc câu, tìm câu trả lời) + tokenizer + cấu hình cho Hugging Face Space, và các báo cáo. Colab có thể xoá file khi hết phiên — tải ngay.

In [ ]:
bundle = Path("/content/foodtrip-assistant-trained.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in (RESULT / "onnx").rglob("*"):
        if item.is_file():
            archive.write(item, Path("onnx") / item.relative_to(RESULT / "onnx"))
    for name in ("metrics.json", "errors.json", "REPORT.md"):
        archive.write(RESULT / name, name)
    RETRIEVER = WORK / "runs/retriever"
    if (RETRIEVER / "onnx/retriever.onnx").exists():
        for item in (RETRIEVER / "onnx").rglob("*"):
            if item.is_file():
                archive.write(item, Path("retriever") / item.relative_to(RETRIEVER / "onnx"))
        for name in ("retriever-metrics.json", "retriever-errors.json", "RETRIEVER.md"):
            archive.write(RETRIEVER / name, name)
files.download(str(bundle))

## Sau khi train
Giải nén **foodtrip-assistant-trained.zip**, chép thư mục `onnx` vào `ml/assistant/space/model` rồi làm theo `ml/assistant/space/README.md` để đưa lên Hugging Face Space.
FoodTrip vẫn chạy bằng bộ luật cho tới khi Space được cấu hình (`ASSISTANT_MODEL_URL`).

## 8. Fine-tune mô hình sinh bằng LoRA
Qwen2.5-1.5B-Instruct, trọng số nền cố định theo revision. Chỉ tối ưu phần trả lời của assistant. Gói tải về gồm adapter và báo cáo; chưa gồm phê duyệt triển khai.

In [ ]:
"""A Colab cell, run after the intent/retriever notebook has created PY and run()."""
from pathlib import Path
import json
import zipfile

SFT = Path('/content/foodtrip-generator')
SFT.mkdir(exist_ok=True)
with zipfile.ZipFile('/content/foodtrip-generator-colab.zip') as archive:
    for item in archive.infolist():
        if not (SFT / item.filename).resolve().is_relative_to(SFT.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(SFT)
run([PY, '-m', 'pip', 'install', '-r', str(SFT / 'requirements-generator.txt')])
run([PY, str(SFT / 'train_generator.py'), '--data', 'sft-data', '--output', 'runs/generator-smoke', '--smoke'], cwd=SFT)
run([PY, str(SFT / 'train_generator.py'), '--data', 'sft-data', '--output', 'runs/generator'], cwd=SFT)
print((SFT / 'runs/generator/GENERATOR.md').read_text())
print((SFT / 'runs/generator/samples.json').read_text())
generator_bundle = Path('/content/foodtrip-generator-trained.zip')
with zipfile.ZipFile(generator_bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for item in (SFT / 'runs/generator').rglob('*'):
        if item.is_file() and 'checkpoints' not in item.parts:
            archive.write(item, item.relative_to(SFT / 'runs/generator'))
files.download(str(generator_bundle))
